# 문제1

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# 데이터 로드
df_psych = pd.read_csv("health_survey.csv")

# 정규화
scaler = StandardScaler()
scaled_features = scaler.fit_transform(df_psych[["bmi", "sleep_hours", "exercise_freq", "smoke", "alcohol"]])

# Isolation Forest
iso = IsolationForest(contamination=0.05, random_state=42)
anomaly_labels = iso.fit_predict(scaled_features)
df_psych["anomaly"] = (anomaly_labels == -1).astype(int)

# 이상 비율 계산
anomaly_rate = df_psych["anomaly"].mean()
print('anomaly_rate : ', anomaly_rate)

# PCA 시각화
pca = PCA(n_components=2)
pca_result = pca.fit_transform(scaled_features)
df_psych["PC1"] = pca_result[:, 0]
df_psych["PC2"] = pca_result[:, 1]

# 산점도 저장
plt.figure(figsize=(8, 6))
plt.scatter(df_psych["PC1"], df_psych["PC2"], c=df_psych["anomaly"], cmap="coolwarm", edgecolor="k", alpha=0.8)
plt.title("PCA Projection of Psychological Data with Anomalies")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plot_path = "anomaly_scatter.png"
plt.savefig(plot_path)
plt.close()

# 결과 저장
csv_path = "psych_anomaly.csv"
df_psych.to_csv(csv_path, index=False)

---

# 문제2

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import os

# YOLOv5 모델 로드
model = torch.hub.load('ultralytics/yolov5', 'yolov5s', pretrained=True)
model.classes = None  # 전체 클래스

# 이미지 경로 설정
image_dir = "./images/training_scene"
image_files = ['CAM001.png', 'CAM002.png', 'CAM003.png']

# 탐지 결과 저장용 리스트
results_list = []

for img_file in image_files:
    img_path = os.path.join(image_dir, img_file)
    img = cv2.imread(img_path)[:, :, ::-1]  # BGR to RGB
    results = model(img)
    df = results.pandas().xyxy[0]

    # 사람/차량 탐지 개수
    person_count = (df["name"] == "person").sum()
    vehicle_count = df["name"].isin(["car", "truck", "bus"]).sum()

    results_list.append({
        "filename": img_file,
        "person_count": person_count,
        "vehicle_count": vehicle_count
    })

# 결과 정리 및 저장
df_result = pd.DataFrame(results_list)
df_result.to_csv("detection_summary.csv", index=False)

# 산점도 시각화
plt.figure(figsize=(8, 6))
plt.scatter(df_result["person_count"], df_result["vehicle_count"], color="purple", edgecolor="k", alpha=0.8)
plt.xlabel("Person Count")
plt.ylabel("Vehicle Count")
plt.title("Detected People vs Vehicles per Image")
plt.grid(True)
plt.tight_layout()
plt.savefig("person_vs_vehicle_scatter.png")
plt.show()

---

# 문제3

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, root_mean_squared_error
import seaborn as sns
import matplotlob.pyplot as plt
import pandas as pd

# 데이터 로드
df = pd.read_csv("soldier_full_profile.csv")

# 피처 및 정답 분리
features = ["pushup", "run_time", "sleep_quality", "stress_level"]
X = df[features]

# 정규화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# [1] 전투 적합 여부 분류 (Logistic Regression)
# status 인코딩 (fit=1, unfit=0)
le = LabelEncoder()
y_status = le.fit_transform(df["status"])  # fit=1, unfit=0

# 분류 모델 학습
clf = LogisticRegression(random_state=42)
clf.fit(X_scaled, y_status)

# 예측 및 평가
y_status_pred = clf.predict(X_scaled)
acc = accuracy_score(y_status, y_status_pred)
cm = confusion_matrix(y_status, y_status_pred)

print('accuracy score : ', acc)
print('confusion matrix : ', cm)

# [2] 회복 기간 회귀 (Random Forest)
y_recovery = df["recovery_days"]

reg = RandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(X_scaled, y_recovery)

y_recovery_pred = reg.predict(X_scaled)
rmse = root_mean_squared_error(y_recovery, y_recovery_pred)

print("rmse : ", rmse)

# 예측 결과 저장 및 시각화
df_result = df[["soldier_id", "status", "recovery_days"]].copy()
df_result["predicted_status"] = le.inverse_transform(y_status_pred)
df_result["predicted_recovery_days"] = y_recovery_pred.round(1)

# 결과 저장
csv_path = "soldier_dual_prediction.csv"
df_result.to_csv(csv_path, index=False)

# 시각화: status 분류결과에 따른 recovery 예측 분포
plt.figure(figsize=(8, 6))
sns.scatterplot(
    x=df_result["predicted_status"],
    y=df_result["predicted_recovery_days"],
    hue=df_result["status"],
    style=df_result["status"],
    s=100
)
plt.title("Recovery Days Prediction by Status")
plt.xlabel("Predicted Status")
plt.ylabel("Predicted Recovery Days")
plt.grid(True)
plt.tight_layout()
plot_path = "dual_prediction.png"
plt.savefig(plot_path)
plt.close()

- train_test_split() 적용

In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import accuracy_score, confusion_matrix, root_mean_squared_error
from sklearn.model_selection import train_test_split
import pandas as pd

# 데이터 로드
df = pd.read_csv("soldier_full_profile.csv")

# 피처 및 정답 분리
features = ["pushup", "run_time", "sleep_quality", "stress_level"]
X = df[features]

# 정규화
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# [1] 전투 적합 여부 분류 (Logistic Regression)
# status 인코딩 (fit=1, unfit=0)
le = LabelEncoder()
y_status = le.fit_transform(df["status"])  # fit=1, unfit=0

# 분류 모델 학습
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_status, test_size=0.2, random_state=42)

clf = LogisticRegression(random_state=42)
clf.fit(X_train, y_train)

# 예측 및 평가
y_status_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_status_pred)
cm = confusion_matrix(y_test, y_status_pred)

print('accuracy score : ', acc)
print('confusion matrix : ', cm)

# [2] 회복 기간 회귀 (Random Forest)
y_recovery = df["recovery_days"]

X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_recovery, test_size=0.2, random_state=42)

reg = RandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(X_train, y_train)

y_recovery_pred = reg.predict(X_test)
rmse = root_mean_squared_error(y_test, y_recovery_pred)

print("rmse : ", rmse)